<h1 align='center' style='color:green'>Credit Risk Modeling Project</h1>

#### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

##### Load Raw Data

In [8]:
df_customers = pd.read_csv("../DATA/raw/customers.csv")
df_loans = pd.read_csv("../DATA/raw/loans.csv")
df_bureau = pd.read_csv("../DATA/raw/bureau_data.csv")

In [11]:
#how many records do we have
df_customers.shape, df_loans.shape, df_bureau.shape

((50000, 12), (50000, 15), (50000, 8))

In [12]:
df_customers.head(3)

,cust_id,age,gender,marital_status,employment_status,income,number_of_dependants,residence_type,years_at_current_address,city,state,zipcode
0,C00001,44,M,Married,Self-Employed,2586000,3,Owned,27,Delhi,Delhi,110001
1,C00002,38,M,Married,Salaried,1206000,3,Owned,4,Chennai,Tamil Nadu,600001
2,C00003,46,F,Married,Self-Employed,2878000,3,Owned,24,Kolkata,West Bengal,700001


In [13]:
df_loans.head(3)

,loan_id,cust_id,loan_purpose,loan_type,sanction_amount,loan_amount,processing_fee,gst,net_disbursement,loan_tenure_months,principal_outstanding,bank_balance_at_application,disbursal_date,installment_start_dt,default
0,L00001,C00001,Auto,Secured,3004000,2467000,49340.0,444060,1973600,33,1630408,873386,2019-07-24,2019-08-10,False
1,L00002,C00002,Home,Secured,4161000,3883000,77660.0,698940,3106400,30,709309,464100,2019-07-24,2019-08-15,False
2,L00003,C00003,Personal,Unsecured,2401000,2170000,43400.0,390600,1736000,21,1562399,1476042,2019-07-24,2019-08-21,False


In [14]:
df_bureau.head(3)

,cust_id,number_of_open_accounts,number_of_closed_accounts,total_loan_months,delinquent_months,total_dpd,enquiry_count,credit_utilization_ratio
0,C00001,1,1,42,0,0,3,7
1,C00002,3,1,96,12,60,5,4
2,C00003,2,1,82,24,147,6,58


In [20]:
# checking duplicate primary keys before merging

print("Duplicate loan_ids :", df_loans.duplicated("loan_id").sum())

print("Duplicate cust_ids :", df_customers.duplicated("cust_id").sum())

# both datasets are safe for merging

Duplicate loan_ids : 0
Duplicate cust_ids : 0


In [21]:
# taking customer's details on base loan file

df = pd.merge(df_loans, df_customers , on ="cust_id", how ='left')
df.head(2)

,loan_id,cust_id,loan_purpose,loan_type,sanction_amount,loan_amount,processing_fee,gst,net_disbursement,loan_tenure_months,principal_outstanding,bank_balance_at_application,disbursal_date,installment_start_dt,default,age,gender,marital_status,employment_status,income,number_of_dependants,residence_type,years_at_current_address,city,state,zipcode
0,L00001,C00001,Auto,Secured,3004000,2467000,49340.0,444060,1973600,33,1630408,873386,2019-07-24,2019-08-10,False,44,M,Married,Self-Employed,2586000,3,Owned,27,Delhi,Delhi,110001
1,L00002,C00002,Home,Secured,4161000,3883000,77660.0,698940,3106400,30,709309,464100,2019-07-24,2019-08-15,False,38,M,Married,Salaried,1206000,3,Owned,4,Chennai,Tamil Nadu,600001


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 26 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_id                      50000 non-null  str    
 1   cust_id                      50000 non-null  str    
 2   loan_purpose                 50000 non-null  str    
 3   loan_type                    50000 non-null  str    
 4   sanction_amount              50000 non-null  int64  
 5   loan_amount                  50000 non-null  int64  
 6   processing_fee               50000 non-null  float64
 7   gst                          50000 non-null  int64  
 8   net_disbursement             50000 non-null  int64  
 9   loan_tenure_months           50000 non-null  int64  
 10  principal_outstanding        50000 non-null  int64  
 11  bank_balance_at_application  50000 non-null  int64  
 12  disbursal_date               50000 non-null  str    
 13  installment_start_dt       

In [26]:
# making output column int
df["default"] = df["default"].astype("int32")
df["default"].dtypes

dtype('int32')

In [ ]:
# checking class imbalanceness

df["default"].value_counts(normalize=True)*100 #1 : defaulter , 0 : non-defauter

default
0    91.406
1     8.594
Name: proportion, dtype: float64

### Train Test Split

In [30]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df,test_size=0.33,random_state=42,stratify=df["default"])

In [31]:
train_df["default"].value_counts(normalize=True)*100 #1 : defaulter , 0 : non-defauter

default
0    91.40597
1     8.59403
Name: proportion, dtype: float64

In [32]:
test_df["default"].value_counts(normalize=True)*100 #1 : defaulter , 0 : non-defauter

default
0    91.406061
1     8.593939
Name: proportion, dtype: float64

In [34]:
# saving splited raw data 
train_df.to_parquet("../DATA/raw/train_df_raw.parquet")
test_df.to_parquet("../DATA/raw/test_df_raw.parquet")